# RT Notebook 19
## Exclusion Generates Residue; Projection Consumes Admissible Distinction

### Theory Series II — Mechanism Isolation

## Core revision

Notebook 18 tested:

\[
\text{projection} \Rightarrow \text{residue}
\]

and found counterexamples.

Notebook 19 tests the stronger mechanistic claim:

\[
\boxed{
R>0
\iff
X>0
}
\]

where:

- \(R\) is residue;
- \(X\) is exclusion of at least one admissible distinction.

Projection itself is treated as **consumption/reorganization of admissible distinction**.
Residue is attributed specifically to **exclusion**, not to projection as such.

# Primary Hypothesis

\[
H_1:
\quad
R(\Pi)>0
\iff
X(\Pi)>0
\]

> Residue is generated if and only if a projection excludes at least one admissible distinction.

This decomposes into two independently testable directions:

### Sufficiency

\[
X>0 \Rightarrow R>0
\]

If exclusion occurs, residue must occur.

### Necessity

\[
R>0 \Rightarrow X>0
\]

If residue occurs, some exclusion must have occurred.

## Falsification conditions

The biconditional fails if either counterexample exists:

1. **Excluded but zero residue**
   \[
   X>0 \land R=0
   \]

2. **Residue without exclusion**
   \[
   R>0 \land X=0
   \]

A single admissible counterexample falsifies the corresponding direction
within the tested model class.

# Deliverables

This notebook produces:

- `projection_records.csv`
- `mechanism_classification.csv`
- `biconditional_test_results.csv`
- `counterexamples_exclusion_without_residue.csv`
- `counterexamples_residue_without_exclusion.csv`
- `exclusion_component_rates.csv`
- `residue_by_exclusion_count.csv`
- `matched_projection_pairs.csv`
- `figure_residue_vs_exclusion.png`
- `figure_mechanism_confusion_matrix.png`
- `figure_residue_by_exclusion_count.png`
- `findings19.json`
- `run_manifest19.json`
- `RT_Notebook_19_outputs.zip`

# Operational Model

A relational organization is:

\[
O=(V,\sigma,E,\rho,\tau)
\]

with:

- \(V\): members;
- \(\sigma\): orientation labels;
- \(E\): relational edges;
- \(\rho\): optional reference member;
- \(\tau\): domain type.

A projection:

\[
\Pi:V_s\rightarrow V_t
\]

is total and deterministic.

## Exclusion vector

\[
X(\Pi)=
(X_{\mathrm{member}},
X_{\mathrm{orientation}},
X_{\mathrm{relation}},
X_{\mathrm{reference}},
X_{\mathrm{alternative}})
\]

where:

- member exclusion: source members identified or dropped from distinct target realization;
- orientation exclusion: incompatible source orientations collapse;
- relation exclusion: source relations do not survive;
- reference exclusion: source reference is lost or overwritten;
- alternative exclusion: admissible source distinctions map to the same realized target state.

## Residue vector

\[
R(\Pi)=
(R_{\mathrm{collision}},
R_{\mathrm{orientation}},
R_{\mathrm{relation}},
R_{\mathrm{reference}},
R_{\mathrm{inverse}})
\]

The notebook tests the biconditional using multiple residue criteria to avoid
definition drift.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations, product
from pathlib import Path
from typing import Iterable, Optional, Sequence, Tuple, List, Dict
from collections import defaultdict
import hashlib
import json
import os
import platform
import random
import statistics
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 190019
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("outputs_notebook19")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output directory:", OUTPUT_DIR.resolve())

## 1. Domain definitions

In [ ]:
@dataclass(frozen=True)
class DomainType:
    name: str
    dof: int
    orientation_required: bool
    relations_required: bool
    reference_required: bool


DOMAIN_TYPES = {
    "core": DomainType("core", 0, False, False, False),
    "ordinal": DomainType("ordinal", 1, True, False, False),
    "gradient": DomainType("gradient", 2, True, True, True),
    "temporal": DomainType("temporal", 3, True, True, True),
}


@dataclass(frozen=True)
class Organization:
    domain: str
    size: int
    orientation: Tuple[int, ...]
    edges: Tuple[Tuple[int, int], ...]
    reference: Optional[int]

    def uid(self) -> str:
        raw = json.dumps({
            "domain": self.domain,
            "size": self.size,
            "orientation": self.orientation,
            "edges": self.edges,
            "reference": self.reference,
        }, sort_keys=True).encode("utf-8")
        return hashlib.sha256(raw).hexdigest()[:16]


def all_edges(n: int) -> Tuple[Tuple[int, int], ...]:
    return tuple(combinations(range(n), 2))


def powerset_edges(n: int) -> Iterable[Tuple[Tuple[int, int], ...]]:
    candidates = all_edges(n)
    for mask in range(1 << len(candidates)):
        yield tuple(
            candidates[i]
            for i in range(len(candidates))
            if mask & (1 << i)
        )


def generate_organizations(domain: str, n: int) -> Iterable[Organization]:
    dtype = DOMAIN_TYPES[domain]

    orientation_space = (
        product((-1, 1), repeat=n)
        if dtype.orientation_required
        else [tuple(0 for _ in range(n))]
    )

    edge_space = list(powerset_edges(n)) if dtype.relations_required else [tuple()]
    reference_space = range(n) if dtype.reference_required else [None]

    for orientation in orientation_space:
        for edges in edge_space:
            for reference in reference_space:
                yield Organization(
                    domain=domain,
                    size=n,
                    orientation=tuple(orientation),
                    edges=tuple(sorted(edges)),
                    reference=reference,
                )


for domain in DOMAIN_TYPES:
    for n in (1, 2, 3):
        count = sum(1 for _ in generate_organizations(domain, n))
        print(f"{domain:8s} n={n}: {count}")

## 2. Projection mechanics

In [ ]:
def all_total_maps(n_source: int, n_target: int):
    return product(range(n_target), repeat=n_source)


def is_injective(mapping: Tuple[int, ...]) -> bool:
    return len(set(mapping)) == len(mapping)


def is_surjective(mapping: Tuple[int, ...], n_target: int) -> bool:
    return set(mapping) == set(range(n_target))


def projected_orientation(
    source: Organization,
    mapping: Tuple[int, ...],
    n_target: int,
) -> Tuple[Optional[int], ...]:
    buckets: List[List[int]] = [[] for _ in range(n_target)]

    for source_index, target_index in enumerate(mapping):
        buckets[target_index].append(source.orientation[source_index])

    result: List[Optional[int]] = []

    for values in buckets:
        if not values:
            result.append(None)
        elif all(v == values[0] for v in values):
            result.append(values[0])
        else:
            result.append(0)

    return tuple(result)


def projected_edges(
    source: Organization,
    mapping: Tuple[int, ...],
) -> set[Tuple[int, int]]:
    result = set()

    for a, b in source.edges:
        x, y = mapping[a], mapping[b]
        if x != y:
            result.add(tuple(sorted((x, y))))

    return result

## 3. Exclusion operationalization

In [ ]:
@dataclass(frozen=True)
class ExclusionVector:
    member: int
    orientation: int
    relation: int
    reference: int
    alternative: int

    @property
    def total(self) -> int:
        return (
            self.member
            + self.orientation
            + self.relation
            + self.reference
            + self.alternative
        )

    @property
    def occurred(self) -> bool:
        return self.total > 0


def compute_exclusion(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> ExclusionVector:

    # Source members that cease to remain distinct.
    member_exclusion = source.size - len(set(mapping))

    # Conflicting source orientations collapsed into one target location.
    orientation_exclusion = 0
    buckets = defaultdict(list)
    for s, t in enumerate(mapping):
        buckets[t].append(source.orientation[s])

    for values in buckets.values():
        if len(set(values)) > 1:
            orientation_exclusion += len(values) - 1

    # Source relations lost because endpoints collapse or target omits the image edge.
    image_edges = projected_edges(source, mapping)
    collapsed_edges = sum(
        1 for a, b in source.edges
        if mapping[a] == mapping[b]
    )
    omitted_image_edges = len(image_edges - set(target.edges))
    relation_exclusion = collapsed_edges + omitted_image_edges

    # Reference excluded when it has no preserved target counterpart.
    if source.reference is None:
        reference_exclusion = 0
    elif target.reference is None:
        reference_exclusion = 1
    else:
        reference_exclusion = int(
            mapping[source.reference] != target.reference
        )

    # Alternative distinction count lost through many-to-one fibers.
    alternative_exclusion = sum(
        max(0, len(values) - 1)
        for values in buckets.values()
    )

    return ExclusionVector(
        member=member_exclusion,
        orientation=orientation_exclusion,
        relation=relation_exclusion,
        reference=reference_exclusion,
        alternative=alternative_exclusion,
    )

## 4. Residue operationalization

In [ ]:
@dataclass(frozen=True)
class ResidueVector:
    collision: int
    orientation: int
    relation: int
    reference: int
    inverse: int

    @property
    def information_total(self) -> int:
        return (
            self.collision
            + self.orientation
            + self.relation
            + self.inverse
        )

    @property
    def structural_total(self) -> int:
        return self.orientation + self.relation

    @property
    def total(self) -> int:
        return (
            self.collision
            + self.orientation
            + self.relation
            + self.reference
            + self.inverse
        )


def compute_residue(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> ResidueVector:

    collision = source.size - len(set(mapping))

    p_orientation = projected_orientation(
        source,
        mapping,
        target.size,
    )

    orientation_residue = 0
    for index, projected_value in enumerate(p_orientation):
        if projected_value is None:
            if DOMAIN_TYPES[target.domain].orientation_required:
                orientation_residue += 1
        elif projected_value == 0:
            orientation_residue += 1
        elif projected_value != target.orientation[index]:
            orientation_residue += 1

    relation_residue = len(
        projected_edges(source, mapping)
        .symmetric_difference(set(target.edges))
    )

    if source.reference is None and target.reference is None:
        reference_residue = 0
    elif source.reference is None or target.reference is None:
        reference_residue = 1
    else:
        reference_residue = int(
            mapping[source.reference] != target.reference
        )

    inverse_residue = int(
        not (
            source.size == target.size
            and is_injective(mapping)
            and is_surjective(mapping, target.size)
        )
    )

    return ResidueVector(
        collision=collision,
        orientation=orientation_residue,
        relation=relation_residue,
        reference=reference_residue,
        inverse=inverse_residue,
    )

## 5. Admissibility

In [ ]:
def weakly_admissible(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> bool:
    return (
        source.domain != target.domain
        and len(mapping) == source.size
        and all(0 <= x < target.size for x in mapping)
    )


def strongly_admissible(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> bool:
    if not weakly_admissible(source, target, mapping):
        return False

    if not is_surjective(mapping, target.size):
        return False

    if source.reference is not None and target.reference is not None:
        if mapping[source.reference] != target.reference:
            return False

    p_orientation = projected_orientation(
        source,
        mapping,
        target.size,
    )

    for index, value in enumerate(p_orientation):
        if value is not None and value != 0:
            if target.orientation[index] != value:
                return False

    # Target cannot contain a relation unsupported by projected source relation.
    if not set(target.edges).issubset(projected_edges(source, mapping)):
        return False

    return True

## 6. Campaign configuration

In [ ]:
CONFIG = {
    "source_domains": ["core", "ordinal", "gradient"],
    "target_domains": ["ordinal", "gradient", "temporal"],
    "source_sizes": [1, 2, 3],
    "target_sizes": [1, 2, 3],
    "target_block_limit": 500,
    "seed": SEED,
}

DOMAIN_PAIRS = [
    (source, target)
    for source in CONFIG["source_domains"]
    for target in CONFIG["target_domains"]
    if source != target
]

print("Domain pairs:", DOMAIN_PAIRS)

In [ ]:
def deterministic_subset(
    organizations: Sequence[Organization],
    limit: int,
) -> List[Organization]:
    if len(organizations) <= limit:
        return list(organizations)

    ordered = sorted(organizations, key=lambda x: x.uid())
    step = len(ordered) / limit
    indices = sorted({
        min(len(ordered) - 1, int(i * step))
        for i in range(limit)
    })
    return [ordered[i] for i in indices]


def campaign_blocks():
    for source_domain, target_domain in DOMAIN_PAIRS:
        for source_size in CONFIG["source_sizes"]:
            for target_size in CONFIG["target_sizes"]:

                source_orgs = list(
                    generate_organizations(source_domain, source_size)
                )

                all_target_orgs = list(
                    generate_organizations(target_domain, target_size)
                )

                target_orgs = deterministic_subset(
                    all_target_orgs,
                    CONFIG["target_block_limit"],
                )

                yield {
                    "source_domain": source_domain,
                    "target_domain": target_domain,
                    "source_size": source_size,
                    "target_size": target_size,
                    "source_orgs": source_orgs,
                    "target_orgs": target_orgs,
                    "target_total": len(all_target_orgs),
                    "target_used": len(target_orgs),
                    "mode": (
                        "exhaustive"
                        if len(target_orgs) == len(all_target_orgs)
                        else "deterministic_stratified"
                    ),
                }

## 7. Execute campaign

In [ ]:
records = []
block_manifest = []
start = time.time()

for block_index, block in enumerate(campaign_blocks(), start=1):
    local_count = 0
    strong_count = 0

    for source in block["source_orgs"]:
        mappings = list(
            all_total_maps(
                block["source_size"],
                block["target_size"],
            )
        )

        for target in block["target_orgs"]:
            for mapping in mappings:
                if not weakly_admissible(source, target, mapping):
                    continue

                exclusion = compute_exclusion(source, target, mapping)
                residue = compute_residue(source, target, mapping)
                strong = strongly_admissible(source, target, mapping)

                records.append({
                    "source_domain": source.domain,
                    "target_domain": target.domain,
                    "source_size": source.size,
                    "target_size": target.size,
                    "source_uid": source.uid(),
                    "target_uid": target.uid(),
                    "mapping": json.dumps(mapping),
                    "strongly_admissible": strong,

                    "member_exclusion": exclusion.member,
                    "orientation_exclusion": exclusion.orientation,
                    "relation_exclusion": exclusion.relation,
                    "reference_exclusion": exclusion.reference,
                    "alternative_exclusion": exclusion.alternative,
                    "exclusion_total": exclusion.total,
                    "exclusion_occurred": exclusion.occurred,

                    "collision_residue": residue.collision,
                    "orientation_residue": residue.orientation,
                    "relation_residue": residue.relation,
                    "reference_residue": residue.reference,
                    "inverse_residue": residue.inverse,
                    "information_residue": residue.information_total,
                    "structural_residue": residue.structural_total,
                    "total_residue": residue.total,

                    "information_residue_occurred": residue.information_total > 0,
                    "structural_residue_occurred": residue.structural_total > 0,
                    "total_residue_occurred": residue.total > 0,
                })

                local_count += 1
                strong_count += int(strong)

    block_manifest.append({
        "block_index": block_index,
        "source_domain": block["source_domain"],
        "target_domain": block["target_domain"],
        "source_size": block["source_size"],
        "target_size": block["target_size"],
        "source_organizations": len(block["source_orgs"]),
        "target_organizations_total": block["target_total"],
        "target_organizations_used": block["target_used"],
        "mode": block["mode"],
        "projection_records": local_count,
        "strong_projection_records": strong_count,
    })

elapsed = time.time() - start
df = pd.DataFrame(records)

print(f"Records: {len(df):,}")
print(f"Strongly admissible: {int(df['strongly_admissible'].sum()):,}")
print(f"Elapsed seconds: {elapsed:,.2f}")

## 8. Validation

In [ ]:
VALIDATION = {
    "records_exist": len(df) > 0,
    "all_cross_domain": bool(
        (df["source_domain"] != df["target_domain"]).all()
    ),
    "exclusion_nonnegative": bool(
        (
            df[
                [
                    "member_exclusion",
                    "orientation_exclusion",
                    "relation_exclusion",
                    "reference_exclusion",
                    "alternative_exclusion",
                ]
            ] >= 0
        ).all().all()
    ),
    "residue_nonnegative": bool(
        (
            df[
                [
                    "collision_residue",
                    "orientation_residue",
                    "relation_residue",
                    "reference_residue",
                    "inverse_residue",
                ]
            ] >= 0
        ).all().all()
    ),
    "exclusion_total_consistent": bool(
        (
            df["exclusion_total"]
            ==
            df[
                [
                    "member_exclusion",
                    "orientation_exclusion",
                    "relation_exclusion",
                    "reference_exclusion",
                    "alternative_exclusion",
                ]
            ].sum(axis=1)
        ).all()
    ),
    "residue_total_consistent": bool(
        (
            df["total_residue"]
            ==
            df[
                [
                    "collision_residue",
                    "orientation_residue",
                    "relation_residue",
                    "reference_residue",
                    "inverse_residue",
                ]
            ].sum(axis=1)
        ).all()
    ),
}

for key, passed in VALIDATION.items():
    print(f"{key:32s}: {'PASS' if passed else 'FAIL'}")

assert all(VALIDATION.values()), "Validation failed."

## 9. Biconditional test

In [ ]:
RESIDUE_CRITERIA = {
    "information": "information_residue_occurred",
    "structural": "structural_residue_occurred",
    "total": "total_residue_occurred",
}

test_rows = []

for admissibility, subset in {
    "weak": df,
    "strong": df[df["strongly_admissible"]],
}.items():

    for criterion, residue_col in RESIDUE_CRITERIA.items():

        excluded = subset["exclusion_occurred"]
        residue = subset[residue_col]

        x1_r1 = int((excluded & residue).sum())
        x1_r0 = int((excluded & ~residue).sum())
        x0_r1 = int((~excluded & residue).sum())
        x0_r0 = int((~excluded & ~residue).sum())

        sufficiency_survives = x1_r0 == 0
        necessity_survives = x0_r1 == 0
        biconditional_survives = sufficiency_survives and necessity_survives

        test_rows.append({
            "admissibility": admissibility,
            "residue_criterion": criterion,
            "tested_projections": len(subset),
            "exclusion_and_residue": x1_r1,
            "exclusion_without_residue": x1_r0,
            "residue_without_exclusion": x0_r1,
            "neither": x0_r0,
            "sufficiency_survives": sufficiency_survives,
            "necessity_survives": necessity_survives,
            "biconditional_survives": biconditional_survives,
            "sufficiency_verdict": (
                "NOT_FALSIFIED"
                if sufficiency_survives
                else "FALSIFIED"
            ),
            "necessity_verdict": (
                "NOT_FALSIFIED"
                if necessity_survives
                else "FALSIFIED"
            ),
            "biconditional_verdict": (
                "NOT_FALSIFIED"
                if biconditional_survives
                else "FALSIFIED"
            ),
        })

test_results = pd.DataFrame(test_rows)
test_results

## 10. Counterexample extraction

In [ ]:
counter_exclusion_without_residue = []
counter_residue_without_exclusion = []

for admissibility, subset in {
    "weak": df,
    "strong": df[df["strongly_admissible"]],
}.items():

    for criterion, residue_col in RESIDUE_CRITERIA.items():

        a = subset[
            subset["exclusion_occurred"]
            & ~subset[residue_col]
        ].copy()

        if not a.empty:
            a.insert(0, "residue_criterion", criterion)
            a.insert(0, "admissibility", admissibility)
            counter_exclusion_without_residue.append(a)

        b = subset[
            ~subset["exclusion_occurred"]
            & subset[residue_col]
        ].copy()

        if not b.empty:
            b.insert(0, "residue_criterion", criterion)
            b.insert(0, "admissibility", admissibility)
            counter_residue_without_exclusion.append(b)

if counter_exclusion_without_residue:
    exclusion_without_residue = pd.concat(
        counter_exclusion_without_residue,
        ignore_index=True,
    )
else:
    exclusion_without_residue = pd.DataFrame()

if counter_residue_without_exclusion:
    residue_without_exclusion = pd.concat(
        counter_residue_without_exclusion,
        ignore_index=True,
    )
else:
    residue_without_exclusion = pd.DataFrame()

print("Exclusion without residue:", len(exclusion_without_residue))
print("Residue without exclusion:", len(residue_without_exclusion))

## 11. Mechanism classification

In [ ]:
def mechanism_class(row):
    x = bool(row["exclusion_occurred"])
    r = bool(row["information_residue_occurred"])

    if x and r:
        return "exclusion_with_residue"
    if x and not r:
        return "exclusion_without_residue"
    if not x and r:
        return "residue_without_exclusion"
    return "lossless_reorganization"


df["mechanism_class"] = df.apply(mechanism_class, axis=1)

mechanism_classification = (
    df.groupby(
        [
            "source_domain",
            "target_domain",
            "strongly_admissible",
            "mechanism_class",
        ]
    )
    .size()
    .reset_index(name="count")
)

mechanism_classification.head(20)

## 12. Exclusion component rates

In [ ]:
exclusion_components = [
    "member_exclusion",
    "orientation_exclusion",
    "relation_exclusion",
    "reference_exclusion",
    "alternative_exclusion",
]

component_rows = []

for admissibility, subset in {
    "weak": df,
    "strong": df[df["strongly_admissible"]],
}.items():

    for component in exclusion_components:
        component_rows.append({
            "admissibility": admissibility,
            "component": component,
            "tested_projections": len(subset),
            "positive_count": int((subset[component] > 0).sum()),
            "positive_rate": float((subset[component] > 0).mean()),
            "mean_magnitude": float(subset[component].mean()),
        })

exclusion_component_rates = pd.DataFrame(component_rows)
exclusion_component_rates

## 13. Residue by exclusion magnitude

In [ ]:
residue_by_exclusion = (
    df[df["strongly_admissible"]]
    .groupby("exclusion_total")
    .agg(
        projection_count=("exclusion_total", "size"),
        mean_information_residue=("information_residue", "mean"),
        median_information_residue=("information_residue", "median"),
        mean_structural_residue=("structural_residue", "mean"),
        mean_total_residue=("total_residue", "mean"),
        information_residue_rate=("information_residue_occurred", "mean"),
        total_residue_rate=("total_residue_occurred", "mean"),
    )
    .reset_index()
)

residue_by_exclusion

## 14. Matched projection analysis

In [ ]:
# Compare projections with identical source/target sizes and domains,
# separating exclusion-free from exclusion-positive cases.

matched_rows = []

group_cols = [
    "source_domain",
    "target_domain",
    "source_size",
    "target_size",
]

for keys, subset in df[df["strongly_admissible"]].groupby(group_cols):
    no_exclusion = subset[~subset["exclusion_occurred"]]
    with_exclusion = subset[subset["exclusion_occurred"]]

    if len(no_exclusion) and len(with_exclusion):
        matched_rows.append({
            "source_domain": keys[0],
            "target_domain": keys[1],
            "source_size": keys[2],
            "target_size": keys[3],
            "no_exclusion_count": len(no_exclusion),
            "with_exclusion_count": len(with_exclusion),
            "mean_information_residue_no_exclusion": float(
                no_exclusion["information_residue"].mean()
            ),
            "mean_information_residue_with_exclusion": float(
                with_exclusion["information_residue"].mean()
            ),
            "difference": float(
                with_exclusion["information_residue"].mean()
                - no_exclusion["information_residue"].mean()
            ),
        })

matched_projection_pairs = pd.DataFrame(matched_rows)
matched_projection_pairs

## 15. Figures

In [ ]:
strong = df[df["strongly_admissible"]]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    strong["exclusion_total"],
    strong["information_residue"],
    alpha=0.25,
)
ax.set_xlabel("Exclusion total")
ax.set_ylabel("Information residue")
ax.set_title("Information residue versus exclusion")
fig.tight_layout()

figure1 = OUTPUT_DIR / "figure_residue_vs_exclusion.png"
fig.savefig(figure1, dpi=180)
plt.show()

In [ ]:
primary = test_results[
    (test_results["admissibility"] == "strong")
    & (test_results["residue_criterion"] == "information")
].iloc[0]

matrix = np.array([
    [
        primary["neither"],
        primary["residue_without_exclusion"],
    ],
    [
        primary["exclusion_without_residue"],
        primary["exclusion_and_residue"],
    ],
])

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No residue", "Residue"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["No exclusion", "Exclusion"])
ax.set_title("Mechanism confusion matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{int(matrix[i, j]):,}", ha="center", va="center")

fig.tight_layout()

figure2 = OUTPUT_DIR / "figure_mechanism_confusion_matrix.png"
fig.savefig(figure2, dpi=180)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    residue_by_exclusion["exclusion_total"],
    residue_by_exclusion["mean_information_residue"],
    marker="o",
)
ax.set_xlabel("Exclusion total")
ax.set_ylabel("Mean information residue")
ax.set_title("Mean residue by exclusion magnitude")
fig.tight_layout()

figure3 = OUTPUT_DIR / "figure_residue_by_exclusion_count.png"
fig.savefig(figure3, dpi=180)
plt.show()

## 16. Interpretation protocol

In [ ]:
def interpret(row):
    suff = row["sufficiency_verdict"]
    nec = row["necessity_verdict"]

    if suff == "NOT_FALSIFIED" and nec == "NOT_FALSIFIED":
        return (
            "The biconditional survives in the tested model class: "
            "exclusion and residue co-occur under this criterion."
        )

    if suff == "FALSIFIED" and nec == "NOT_FALSIFIED":
        return (
            "Exclusion is necessary but not sufficient. Some exclusions "
            "produce no measured residue."
        )

    if suff == "NOT_FALSIFIED" and nec == "FALSIFIED":
        return (
            "Exclusion is sufficient but not necessary. Some residue appears "
            "without measured exclusion."
        )

    return (
        "Both directions fail. Exclusion and residue are correlated but not "
        "equivalent under this operationalization."
    )


test_results["interpretation"] = test_results.apply(interpret, axis=1)

for _, row in test_results.iterrows():
    print(
        f"\n[{row['admissibility']} / {row['residue_criterion']}]\n"
        f"Sufficiency: {row['sufficiency_verdict']}\n"
        f"Necessity: {row['necessity_verdict']}\n"
        f"Biconditional: {row['biconditional_verdict']}\n"
        f"{row['interpretation']}"
    )

## 17. Export deliverables

In [ ]:
projection_records_path = OUTPUT_DIR / "projection_records.csv"
mechanism_classification_path = OUTPUT_DIR / "mechanism_classification.csv"
test_results_path = OUTPUT_DIR / "biconditional_test_results.csv"
counter_x_no_r_path = OUTPUT_DIR / "counterexamples_exclusion_without_residue.csv"
counter_r_no_x_path = OUTPUT_DIR / "counterexamples_residue_without_exclusion.csv"
component_rates_path = OUTPUT_DIR / "exclusion_component_rates.csv"
residue_by_exclusion_path = OUTPUT_DIR / "residue_by_exclusion_count.csv"
matched_pairs_path = OUTPUT_DIR / "matched_projection_pairs.csv"
block_manifest_path = OUTPUT_DIR / "block_manifest.csv"

df.to_csv(projection_records_path, index=False)
mechanism_classification.to_csv(mechanism_classification_path, index=False)
test_results.to_csv(test_results_path, index=False)
exclusion_without_residue.to_csv(counter_x_no_r_path, index=False)
residue_without_exclusion.to_csv(counter_r_no_x_path, index=False)
exclusion_component_rates.to_csv(component_rates_path, index=False)
residue_by_exclusion.to_csv(residue_by_exclusion_path, index=False)
matched_projection_pairs.to_csv(matched_pairs_path, index=False)
pd.DataFrame(block_manifest).to_csv(block_manifest_path, index=False)

print("CSV deliverables written.")

In [ ]:
strong_primary = test_results[
    test_results["admissibility"] == "strong"
].copy()

findings = {
    "notebook": 19,
    "title": "Exclusion Generates Residue",
    "primary_hypothesis": (
        "Residue is generated if and only if a projection excludes at least "
        "one admissible distinction."
    ),
    "causal_revision": {
        "projection_role": (
            "Projection consumes and reorganizes admissible distinction."
        ),
        "exclusion_role": (
            "Exclusion is the candidate generator of residue."
        ),
    },
    "scope": {
        "model_class": "bounded finite relational organizations",
        "source_domains": CONFIG["source_domains"],
        "target_domains": CONFIG["target_domains"],
        "source_sizes": CONFIG["source_sizes"],
        "target_sizes": CONFIG["target_sizes"],
        "admissibility_classes": ["weak", "strong"],
        "residue_criteria": list(RESIDUE_CRITERIA.keys()),
    },
    "strong_admissibility_results": strong_primary[
        [
            "residue_criterion",
            "tested_projections",
            "exclusion_and_residue",
            "exclusion_without_residue",
            "residue_without_exclusion",
            "neither",
            "sufficiency_verdict",
            "necessity_verdict",
            "biconditional_verdict",
            "interpretation",
        ]
    ].to_dict(orient="records"),
    "validation": VALIDATION,
    "decision_rule": (
        "Any exclusion-without-residue case falsifies sufficiency. "
        "Any residue-without-exclusion case falsifies necessity."
    ),
    "limitations": [
        "Finite bounded model class.",
        "Some target blocks may use deterministic stratification.",
        "Exclusion and residue remain operational definitions.",
        "Association does not establish ontological causation.",
        "A surviving biconditional is not a proof beyond the tested model class.",
    ],
    "recommended_next_notebook": {
        "notebook": 20,
        "title": "Exclusion Typology and Residue Generation",
        "question": (
            "Which types of exclusion are independently sufficient for which "
            "types of residue?"
        ),
    },
}

findings_path = OUTPUT_DIR / "findings19.json"
findings_path.write_text(
    json.dumps(findings, indent=2),
    encoding="utf-8",
)

run_manifest = {
    "notebook": 19,
    "seed": SEED,
    "python": sys.version,
    "platform": platform.platform(),
    "elapsed_seconds": elapsed,
    "record_count": len(df),
    "strong_record_count": int(df["strongly_admissible"].sum()),
    "configuration": CONFIG,
    "blocks": block_manifest,
    "artifacts": [
        str(projection_records_path),
        str(mechanism_classification_path),
        str(test_results_path),
        str(counter_x_no_r_path),
        str(counter_r_no_x_path),
        str(component_rates_path),
        str(residue_by_exclusion_path),
        str(matched_pairs_path),
        str(block_manifest_path),
        str(figure1),
        str(figure2),
        str(figure3),
        str(findings_path),
    ],
}

manifest_path = OUTPUT_DIR / "run_manifest19.json"
manifest_path.write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)

print(json.dumps(findings, indent=2))

## 18. Package outputs

In [ ]:
zip_path = OUTPUT_DIR / "RT_Notebook_19_outputs.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path == zip_path:
            continue
        if path.is_file():
            zf.write(path, arcname=path.name)

print("Archive:", zip_path.resolve())
print("Size:", zip_path.stat().st_size, "bytes")

# Final reporting rule

The notebook must report separately:

- whether exclusion is sufficient for residue;
- whether exclusion is necessary for residue;
- whether the full biconditional survives.

The result must not be collapsed into a single narrative conclusion if the two
directions differ.

# Theoretical consequence

If the biconditional survives for information residue, the causal model becomes:

\[
\text{projection}
\rightarrow
\text{consumption/reorganization}
\rightarrow
\text{exclusion}
\rightarrow
\text{residue}
\]

If it fails, Notebook 20 must identify which exclusion types generate which
residue types.